## Table section

In [13]:
from pathlib import Path
from collections import defaultdict

# ========== DYNAMIC POLICY DISCOVERY ==========
BASE_LOG_DIR = "../tb_logs/res_validator"

def find_event_files(base_dir):
    """
    Recursively find all event files in the directory.
    Returns a dictionary mapping policy names to event file paths.
    """
    event_files = {}
    base_path = Path(base_dir)
    
    # Find all event files
    for event_file in base_path.rglob("events.out.tfevents*"):
        event_path = str(event_file)
        event_files[event_path] = event_path
    
    return event_files

def extract_policy_name(event_path):
    """
    Extract a human-readable policy name from the event path.
    Handles patterns like:
    - idle/gym_from_testing_config_1ms_10s_8UE_jam_fixed_5/...
    - dqn/gym_from_testing_config_1ms_10s_8UE_jam_pattern_fixed/...
    - random/gym_from_testing_config_1ms_10s_8UE_jam_fixed_baseline_0/...
    """
    parts = event_path.replace("\\", "/").split("/")
    
    # Extract policy type (idle, dqn, qrdqn, random, aggressive)
    policy_type = None
    config_name = None
    
    for i, part in enumerate(parts):
        if part in ["idle", "dqn", "qrdqn", "random", "aggressive"]:
            policy_type = part
            # Get config name from next directory level if it exists
            if i + 1 < len(parts) and "gym" in parts[i + 1]:
                config_name = parts[i + 1]
            break
    
    if policy_type is None:
        # Fallback: check for direct policy directories
        if "idle__" in event_path:
            return "Idle"
        elif "random__" in event_path:
            return "Random"
        elif "xAppResilientDQNAllocator" in event_path:
            return "DQN"
        elif "xAppResilientQRDQNAllocator" in event_path:
            return "QRDQN"
        elif "aggressive__" in event_path:
            return "Aggressive"
        else:
            return "Unknown"
    
    # Format the name
    policy_map = {
        "idle": "Idle",
        "random": "Random",
        "dqn": "DQN",
        "qrdqn": "QRDQN",
        "aggressive": "Aggressive"
    }
    
    base_name = policy_map.get(policy_type, policy_type.upper())
    
    if config_name:
        # Clean up config name: remove common prefixes
        clean_config = config_name.replace("gym_from_testing_config_1ms_10s_8UE_", "").replace("gym_from_testing_config_1ms_10s_", "")
        # Capitalize first letter of each section
        clean_config = "_".join([s.capitalize() for s in clean_config.split("_")])
        return f"{base_name}_{clean_config}"
    
    return base_name

def build_events_dict(base_dir):
    """
    Build the EVENTS dictionary from discovered event files,
    organized by policy and configuration.
    """
    event_files = find_event_files(base_dir)
    events_by_policy = defaultdict(list)
    
    for event_path in event_files.values():
        policy_name = extract_policy_name(event_path)
        events_by_policy[policy_name].append(event_path)
    
    # Create final EVENTS dict
    events = {}
    for policy, paths in sorted(events_by_policy.items()):
        if len(paths) == 1:
            events[policy] = paths[0]
        else:
            # If multiple paths for same policy, use the most recent (last one typically)
            events[policy] = paths[-1]
    
    return events

# Build the EVENTS dictionary dynamically
EVENTS = build_events_dict(BASE_LOG_DIR)

print(f"Found {len(EVENTS)} unique policies/configurations:")
for policy_name, path in sorted(EVENTS.items()):
    print(f"  - {policy_name}: {path}")

Found 37 unique policies/configurations:
  - Aggressive_Jam_Fixed_10: ../tb_logs/res_validator/aggressive/gym_from_testing_config_1ms_10s_8UE_jam_fixed_10/aggressive__20260122_105830__res18_off0_const0_per-1_dur0/events.out.tfevents.1769097510.elahe-LEGION-T7-34IAS10.1095682.0
  - Aggressive_Jam_Fixed_15: ../tb_logs/res_validator/aggressive/gym_from_testing_config_1ms_10s_8UE_jam_fixed_15/aggressive__20260122_115248__res18_off0_const0_per-1_dur0/events.out.tfevents.1769100768.elahe-LEGION-T7-34IAS10.1099725.0
  - Aggressive_Jam_Fixed_20: ../tb_logs/res_validator/aggressive/gym_from_testing_config_1ms_10s_8UE_jam_fixed_20/aggressive__20260122_124558__res18_off0_const0_per-1_dur0/events.out.tfevents.1769103958.elahe-LEGION-T7-34IAS10.1101090.0
  - Aggressive_Jam_Fixed_5: ../tb_logs/res_validator/aggressive/gym_from_testing_config_1ms_10s_8UE_jam_fixed_5/aggressive__20260122_133844__res18_off0_const0_per-1_dur0/events.out.tfevents.1769107124.elahe-LEGION-T7-34IAS10.1102251.0
  - Aggressiv

In [14]:

def categorize_policies(events_dict):
    """
    Categorize policies into groups based on their type and configuration.
    Useful for organizing comparison tables.
    Checks for more specific patterns first to avoid misclassification.
    """
    categories = defaultdict(list)
    
    for policy_name in events_dict.keys():
        # Check for specific policy types FIRST (before checking for "Random")
        if "QRDQN" in policy_name:
            categories["Learning - QRDQN"].append(policy_name)
        elif "DQN" in policy_name:
            categories["Learning - DQN"].append(policy_name)
        elif "Aggressive" in policy_name:
            categories["Baseline - Aggressive"].append(policy_name)
        elif "Idle" in policy_name:
            categories["Baseline - Idle"].append(policy_name)
        elif "Random" in policy_name:
            categories["Baseline - Random"].append(policy_name)
        else:
            categories["Other"].append(policy_name)
    
    return categories

# Categorize discovered policies
policy_categories = categorize_policies(EVENTS)

print("\n\nPolicy Categories:")
for category, policies in sorted(policy_categories.items()):
    print(f"\n{category}:")
    for policy in sorted(policies):
        print(f"  ✓ {policy}")




Policy Categories:

Baseline - Aggressive:
  ✓ Aggressive_Jam_Fixed_10
  ✓ Aggressive_Jam_Fixed_15
  ✓ Aggressive_Jam_Fixed_20
  ✓ Aggressive_Jam_Fixed_5
  ✓ Aggressive_Jam_Fixed_Baseline_0
  ✓ Aggressive_Jam_Pattern_Decreasing
  ✓ Aggressive_Jam_Pattern_Fixed
  ✓ Aggressive_Jam_Pattern_Increasing
  ✓ Aggressive_Jam_Pattern_Random

Baseline - Idle:
  ✓ Idle
  ✓ Idle_Jam_Fixed_10
  ✓ Idle_Jam_Fixed_15
  ✓ Idle_Jam_Fixed_20
  ✓ Idle_Jam_Fixed_5
  ✓ Idle_Jam_Fixed_Baseline_0
  ✓ Idle_Jam_Pattern_Decreasing
  ✓ Idle_Jam_Pattern_Fixed
  ✓ Idle_Jam_Pattern_Increasing
  ✓ Idle_Jam_Pattern_Random

Baseline - Random:
  ✓ Random_Jam_Fixed_10
  ✓ Random_Jam_Fixed_15
  ✓ Random_Jam_Fixed_20
  ✓ Random_Jam_Fixed_5
  ✓ Random_Jam_Fixed_Baseline_0
  ✓ Random_Jam_Pattern_Decreasing
  ✓ Random_Jam_Pattern_Fixed
  ✓ Random_Jam_Pattern_Increasing
  ✓ Random_Jam_Pattern_Random

Learning - DQN:
  ✓ DQN_Jam_Fixed_10
  ✓ DQN_Jam_Fixed_15
  ✓ DQN_Jam_Fixed_20
  ✓ DQN_Jam_Fixed_5
  ✓ DQN_Jam_Fixed_Baseline_0

In [15]:
import numpy as np
import pandas as pd
import warnings
from tensorflow.python.summary.summary_iterator import summary_iterator

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ========== LATENCY UNITS CONFIGURATION ==========
LATENCY_UNIT = "both"  # Options: "sec", "ms", or "both"
URLLC_LATENCY_DEADLINE_SEC = 0.001   # 1 ms in seconds
URLLC_LATENCY_DEADLINE_MS = 1.0      # 1 ms

URLLC_LATENCY_DEADLINE = {
    "sec": URLLC_LATENCY_DEADLINE_SEC,
    "ms": URLLC_LATENCY_DEADLINE_MS,
}

JAM_THRESHOLD = 0.99
RECOVERY_HOLD_STEPS = 10
FLOAT_FMT = "{:.3f}"

def load_tb_scalars(event_file):
    data = {}
    for e in summary_iterator(event_file):
        if not e.summary.value:
            continue
        for v in e.summary.value:
            data.setdefault(v.tag, []).append((e.step, v.simple_value))
    return {
        tag: pd.DataFrame(vals, columns=["step", "value"]).sort_values("step")
        for tag, vals in data.items()
    }

def has_tags(tb, tags):
    return all(tag in tb for tag in tags)

def print_table(name, df, unit=""):
    unit_str = f" ({unit})" if unit else ""
    print("\n" + "=" * 100)
    print(f"TABLE: {name.upper()}{unit_str}")
    print("=" * 100)
    print(df.to_string(float_format=lambda x: FLOAT_FMT.format(x)))

def recovery_time(lat, jam_mask, deadline):
    """Time from jamming end until latency stays below deadline for hold_steps"""
    idx = np.where(jam_mask)[0]
    if len(idx) == 0:
        return np.nan
    jam_end = idx[-1]
    below = lat <= deadline
    for i in range(jam_end + 1, len(below) - RECOVERY_HOLD_STEPS):
        if np.all(below[i:i + RECOVERY_HOLD_STEPS]):
            return i - jam_end
    return np.inf

# ========== SECONDS-BASED TABLE FUNCTIONS ==========

def table_performance_sec(tb):
    required = ["latency/urllc", "throughput/embb_mbps", "reward/total"]
    if not has_tags(tb, required):
        return None
    lat = tb["latency/urllc"]["value"]
    thr = tb["throughput/embb_mbps"]["value"]
    rew = tb["reward/total"]["value"]
    return {
        "avg_urllc_latency": lat.mean(),
        "latency_violation_rate": np.mean(lat > URLLC_LATENCY_DEADLINE["sec"]),
        "avg_embb_throughput": thr.mean(),
        "avg_reward": rew.mean(),
    }

def table_resilience_sec(tb):
    required = ["latency/urllc", "prb/effective_ratio", "obs/reserve_used_prb"]
    if not has_tags(tb, required):
        return None
    df = tb["latency/urllc"].merge(tb["prb/effective_ratio"], on="step", how="inner")
    lat = df["value_x"].values
    jam = df["value_y"].values < JAM_THRESHOLD
    return {
        "max_latency_spike": lat.max(),
        "recovery_time": recovery_time(lat, jam, URLLC_LATENCY_DEADLINE["sec"]),
        "avg_reserve_prb_used": tb["obs/reserve_used_prb"]["value"].mean(),
    }

def table_jamming_split_sec(tb):
    required = ["latency/urllc", "prb/effective_ratio", "obs/urllc_queue", "obs/reserve_used_prb"]
    if not has_tags(tb, required):
        return None
    df = (tb["latency/urllc"].merge(tb["prb/effective_ratio"], on="step", how="inner")
          .merge(tb["obs/urllc_queue"], on="step", how="inner"))
    lat = df["value_x"]
    jam = df["value_y"] < JAM_THRESHOLD
    queue = df["value"]
    r = tb["obs/reserve_used_prb"]["value"]
    return {
        "lat_no_jam": lat[~jam].mean(),
        "lat_jam": lat[jam].mean(),
        "delta_latency": lat[jam].mean() - lat[~jam].mean(),
        "queue_jam": queue[jam].mean(),
        "reserve_used_jam": r.mean(),
    }

def table_reserve_efficiency_sec(tb):
    required = ["latency/urllc", "prb/effective_ratio", "obs/reserve_used_prb"]
    if not has_tags(tb, required):
        return None
    df = tb["latency/urllc"].merge(tb["prb/effective_ratio"], on="step", how="inner")
    lat = df["value_x"]
    jam = df["value_y"] < JAM_THRESHOLD
    r = tb["obs/reserve_used_prb"]["value"]
    violations = np.mean(lat[jam] > URLLC_LATENCY_DEADLINE["sec"])
    return {
        "reserve_used_overall": r.mean(),
        "reserve_used_no_jam": r[~jam].mean(),
        "reserve_used_jam": r[jam].mean(),
        "violation_rate_jam": violations,
        "max_latency_jam": lat[jam].max(),
    }

# ========== MILLISECONDS-BASED TABLE FUNCTIONS ==========

def table_performance_ms(tb):
    required = ["latency/urllc", "throughput/embb_mbps", "reward/total"]
    if not has_tags(tb, required):
        return None
    lat = tb["latency/urllc"]["value"] * 1000.0
    thr = tb["throughput/embb_mbps"]["value"]
    rew = tb["reward/total"]["value"]
    return {
        "avg_urllc_latency": lat.mean(),
        "latency_violation_rate": np.mean(lat > URLLC_LATENCY_DEADLINE["ms"]),
        "avg_embb_throughput": thr.mean(),
        "avg_reward": rew.mean(),
    }

def table_resilience_ms(tb):
    required = ["latency/urllc", "prb/effective_ratio", "obs/reserve_used_prb"]
    if not has_tags(tb, required):
        return None
    df = tb["latency/urllc"].merge(tb["prb/effective_ratio"], on="step", how="inner")
    lat = df["value_x"].values * 1000.0
    jam = df["value_y"].values < JAM_THRESHOLD
    return {
        "max_latency_spike": lat.max(),
        "recovery_time": recovery_time(lat, jam, URLLC_LATENCY_DEADLINE["ms"]),
        "avg_reserve_prb_used": tb["obs/reserve_used_prb"]["value"].mean(),
    }

def table_jamming_split_ms(tb):
    required = ["latency/urllc", "prb/effective_ratio", "obs/urllc_queue", "obs/reserve_used_prb"]
    if not has_tags(tb, required):
        return None
    df = (tb["latency/urllc"].merge(tb["prb/effective_ratio"], on="step", how="inner")
          .merge(tb["obs/urllc_queue"], on="step", how="inner"))
    lat = df["value_x"] * 1000.0
    jam = df["value_y"] < JAM_THRESHOLD
    queue = df["value"]
    r = tb["obs/reserve_used_prb"]["value"]
    return {
        "lat_no_jam": lat[~jam].mean(),
        "lat_jam": lat[jam].mean(),
        "delta_latency": lat[jam].mean() - lat[~jam].mean(),
        "queue_jam": queue[jam].mean(),
        "reserve_used_jam": r.mean(),
    }

def table_reserve_efficiency_ms(tb):
    required = ["latency/urllc", "prb/effective_ratio", "obs/reserve_used_prb"]
    if not has_tags(tb, required):
        return None
    df = tb["latency/urllc"].merge(tb["prb/effective_ratio"], on="step", how="inner")
    lat = df["value_x"] * 1000.0
    jam = df["value_y"] < JAM_THRESHOLD
    r = tb["obs/reserve_used_prb"]["value"]
    violations = np.mean(lat[jam] > URLLC_LATENCY_DEADLINE["ms"])
    return {
        "reserve_used_overall": r.mean(),
        "reserve_used_no_jam": r[~jam].mean(),
        "reserve_used_jam": r[jam].mean(),
        "violation_rate_jam": violations,
        "max_latency_jam": lat[jam].max(),
    }


In [16]:
STATE_TAGS = [
    "state/embb_demand_norm",
    "state/urllc_demand_norm",
    "state/embb_queue_norm",
    "state/urllc_queue_norm",
    "state/embb_thr_norm",
    "state/urllc_thr_norm",
    "state/urllc_lat_excess_norm",
    "state/effective_prb_ratio",
    "state/reserve_used_norm",
]

def table_state_stats(tb):
    """Unit-agnostic: returns state statistics"""
    if not has_tags(tb, STATE_TAGS):
        return None

    row = {}
    for tag in STATE_TAGS:
        v = tb[tag]["value"]
        row[f"{tag}_mean"] = v.mean()
        row[f"{tag}_std"] = v.std()
    return row

def table_actions(tb):
    """Unit-agnostic: returns action statistics"""
    if "action/index" not in tb:
        return None
    
    a = tb["action/index"]["value"]
    da = np.abs(np.diff(a))

    return {
        "action_variance": a.var(),
        "action_change_rate": np.mean(da > 0),
        "mean_abs_action_change": da.mean(),
    }

def table_satisfaction(tb):
    """Unit-agnostic: returns satisfaction metrics"""
    required = ["satisfaction/URLLC_capped", "satisfaction/eMBB_capped", "satisfaction/URLLC_raw"]
    if not has_tags(tb, required):
        return None
    
    return {
        "urllc_sat_capped": tb["satisfaction/URLLC_capped"]["value"].mean(),
        "embb_sat_capped": tb["satisfaction/eMBB_capped"]["value"].mean(),
        "urllc_sat_worst": tb["satisfaction/URLLC_raw"]["value"].min(),
    }

def table_episode(tb):
    """Unit-agnostic: returns episode statistics"""
    required = ["episode/return", "episode/length"]
    if not has_tags(tb, required):
        return None
    
    r = tb["episode/return"]["value"]
    l = tb["episode/length"]["value"]

    return {
        "return_mean": r.mean(),
        "return_std": r.std(),
        "return_p10": np.percentile(r, 10),
        "episode_length_mean": l.mean(),
    }

def table_correlation(tb):
    """Unit-agnostic: returns correlation metrics"""
    required = [
        "latency/urllc",
        "prb/effective_ratio",
        "state/urllc_queue_norm",
        "state/reserve_used_norm",
    ]
    if not has_tags(tb, required):
        return None

    df = tb["latency/urllc"].merge(
        tb["prb/effective_ratio"], on="step", how="inner"
    )

    jam = df["value_y"] < JAM_THRESHOLD
    lat = df["value_x"][jam]

    df2 = (
        tb["state/urllc_queue_norm"]
        .merge(tb["state/reserve_used_norm"], on="step", how="inner")
    )
    q = df2["value_x"][jam]
    r = df2["value_y"][jam]

    # Handle edge case: need at least 2 points for correlation
    if len(lat) < 2 or len(q) < 2 or len(r) < 2:
        return {
            "corr_queue_latency": np.nan,
            "corr_reserve_latency": np.nan,
        }

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        corr_q_lat = np.corrcoef(q, lat)[0, 1] if len(q) > 1 else np.nan
        corr_r_lat = np.corrcoef(r, lat)[0, 1] if len(r) > 1 else np.nan

    return {
        "corr_queue_latency": corr_q_lat,
        "corr_reserve_latency": corr_r_lat,
    }


In [17]:
# Additional utility functions (kept for reference, not currently used in TABLE_CONFIGS)

def table_jamming_split(tb):
    """Helper variant - uses the _sec version in actual table generation"""
    required = [
        "latency/urllc",
        "prb/effective_ratio",
        "obs/urllc_queue",
        "obs/reserve_used_prb",
    ]
    if not has_tags(tb, required):
        return None

    df = (
        tb["latency/urllc"]
        .merge(tb["prb/effective_ratio"], on="step", how="inner")
        .merge(tb["obs/urllc_queue"], on="step", how="inner")
    )

    lat = df["value_x"]
    jam = df["value_y"] < JAM_THRESHOLD
    queue = df["value"]

    r = tb["obs/reserve_used_prb"]["value"]

    return {
        "lat_no_jam": lat[~jam].mean(),
        "lat_jam": lat[jam].mean(),
        "delta_latency": lat[jam].mean() - lat[~jam].mean(),
        "queue_jam": queue[jam].mean(),
        "reserve_used_jam": r.mean(),
    }


In [18]:
# Additional eMBB analysis functions (for future use if needed)

def table_embb_throughput(tb):
    """eMBB throughput statistics - can be added to TABLE_CONFIGS if needed"""
    if "throughput/embb_mbps" not in tb:
        return None

    v = tb["throughput/embb_mbps"]["value"]
    return {
        "embb_thr_mean": v.mean(),
        "embb_thr_std": v.std(),
    }

def table_embb_jam_split(tb):
    """eMBB throughput split by jamming state - can be added to TABLE_CONFIGS if needed"""
    required = [
        "throughput/embb_mbps",
        "prb/effective_ratio",
    ]
    if not has_tags(tb, required):
        return None

    df = tb["throughput/embb_mbps"].merge(
        tb["prb/effective_ratio"], on="step", how="inner"
    )

    thr = df["value_x"]
    jam = df["value_y"] < JAM_THRESHOLD

    return {
        "embb_thr_no_jam": thr[~jam].mean(),
        "embb_thr_jam": thr[jam].mean(),
        "embb_thr_drop": thr[jam].mean() - thr[~jam].mean(),
    }

def table_embb_efficiency(tb):
    """eMBB spectral efficiency - can be added to TABLE_CONFIGS if needed"""
    required = [
        "obs/embb_throughput",
        "obs/embb_base_achievable_throughput",
        "prb/effective_ratio",
    ]
    if not has_tags(tb, required):
        return None

    df = (
        tb["obs/embb_throughput"]
        .merge(tb["obs/embb_base_achievable_throughput"], on="step", how="inner")
        .merge(tb["prb/effective_ratio"], on="step", how="inner")
    )

    thr = df["value_x"]
    base = df["value_y"]
    eff = thr / np.maximum(base, 1e-6)
    jam = df["value"] < JAM_THRESHOLD

    return {
        "embb_eff_mean": eff.mean(),
        "embb_eff_jam": eff[jam].mean(),
    }

def table_embb_state(tb):
    """eMBB state statistics - can be added to TABLE_CONFIGS if needed"""
    if "state/embb_thr_norm" not in tb:
        return None

    v = tb["state/embb_thr_norm"]["value"]
    return {
        "embb_thr_norm_mean": v.mean(),
        "embb_thr_norm_std": v.std(),
    }


In [19]:
# EVENTS is now built dynamically from the previous cell

# Table functions: (name, sec_function, ms_function, unit_agnostic)
# If unit_agnostic=True, only one function is needed
TABLE_CONFIGS = [
    ("performance", table_performance_sec, table_performance_ms, False),
    ("resilience", table_resilience_sec, table_resilience_ms, False),
    ("jamming_split", table_jamming_split_sec, table_jamming_split_ms, False),
    ("reserve_efficiency", table_reserve_efficiency_sec, table_reserve_efficiency_ms, False),
    ("state_stats", table_state_stats, None, True),
    ("actions", table_actions, None, True),
    ("satisfaction", table_satisfaction, None, True),
    ("episode", table_episode, None, True),
    ("correlation", table_correlation, None, True),
]

# Generate tables for both units
tables_sec = {name: [] for name, _, _, _ in TABLE_CONFIGS}
tables_ms = {name: [] for name, _, _, _ in TABLE_CONFIGS}

print(f"\nProcessing {len(EVENTS)} policies...")
failed_policies = []

for policy, path in EVENTS.items():
    try:
        tb = load_tb_scalars(path)
        
        for name, fn_sec, fn_ms, is_agnostic in TABLE_CONFIGS:
            if is_agnostic:
                # Unit-agnostic: generate once
                row = fn_sec(tb)
                if row is not None:
                    row["policy"] = policy
                    tables_sec[name].append(row)
                    tables_ms[name].append(row.copy())
            else:
                # Unit-specific: generate for both sec and ms
                row_sec = fn_sec(tb)
                row_ms = fn_ms(tb)
                
                if row_sec is not None:
                    row_sec["policy"] = policy
                    tables_sec[name].append(row_sec)
                
                if row_ms is not None:
                    row_ms["policy"] = policy
                    tables_ms[name].append(row_ms)
        
        print(f"  ✓ {policy}")
    except Exception as e:
        print(f"  ✗ {policy}: {str(e)[:80]}")
        failed_policies.append((policy, str(e)))

if failed_policies:
    print(f"\n⚠ {len(failed_policies)} policies failed to process:")
    for policy, error in failed_policies:
        print(f"  - {policy}: {error[:60]}...")



Processing 37 policies...
  ✓ Aggressive_Jam_Fixed_10
  ✓ Aggressive_Jam_Fixed_15
  ✓ Aggressive_Jam_Fixed_20
  ✓ Aggressive_Jam_Fixed_5
  ✓ Aggressive_Jam_Fixed_Baseline_0
  ✓ Aggressive_Jam_Pattern_Decreasing
  ✓ Aggressive_Jam_Pattern_Fixed
  ✓ Aggressive_Jam_Pattern_Increasing
  ✓ Aggressive_Jam_Pattern_Random
  ✓ DQN_Jam_Fixed_10
  ✓ DQN_Jam_Fixed_15
  ✓ DQN_Jam_Fixed_20
  ✓ DQN_Jam_Fixed_5
  ✓ DQN_Jam_Fixed_Baseline_0
  ✓ DQN_Jam_Pattern_Decreasing
  ✓ DQN_Jam_Pattern_Fixed
  ✓ DQN_Jam_Pattern_Increasing
  ✓ DQN_Jam_Pattern_Random
  ✓ Idle
  ✓ Idle_Jam_Fixed_10
  ✓ Idle_Jam_Fixed_15
  ✓ Idle_Jam_Fixed_20
  ✓ Idle_Jam_Fixed_5
  ✓ Idle_Jam_Fixed_Baseline_0
  ✓ Idle_Jam_Pattern_Decreasing
  ✓ Idle_Jam_Pattern_Fixed
  ✓ Idle_Jam_Pattern_Increasing
  ✓ Idle_Jam_Pattern_Random
  ✓ Random_Jam_Fixed_10
  ✓ Random_Jam_Fixed_15
  ✓ Random_Jam_Fixed_20
  ✓ Random_Jam_Fixed_5
  ✓ Random_Jam_Fixed_Baseline_0
  ✓ Random_Jam_Pattern_Decreasing
  ✓ Random_Jam_Pattern_Fixed
  ✓ Random_Jam_Patter

In [20]:
def save_and_display_tables(tables_dict, unit_name, unit_str):
    """Helper function to save and display all tables for a given unit"""
    print("\n\n" + "="*120)
    print(f"GENERATING TABLES FOR {unit_name.upper()}")
    print("="*120)
    
    generated_count = 0
    empty_count = 0
    
    for name, rows in tables_dict.items():
        if len(rows) == 0:
            print(f"[SKIP] Table '{name}' is empty")
            empty_count += 1
            continue
        
        df = pd.DataFrame(rows).set_index("policy")
        # Sort dataframe by index for consistent output
        df = df.sort_index()
        
        # Save to CSV
        csv_filename = f"table_{name}_{unit_str}.csv"
        df.to_csv(csv_filename)
        
        print_table(name, df, unit=unit_name)
        print(f"  → Saved to: {csv_filename}")
        generated_count += 1
    
    print(f"\n📊 Summary: Generated {generated_count} tables, Skipped {empty_count} empty tables")
    return generated_count, empty_count

# Generate and save tables
sec_count, sec_empty = save_and_display_tables(tables_sec, "seconds", "sec")
ms_count, ms_empty = save_and_display_tables(tables_ms, "milliseconds", "ms")

print("\n\n" + "="*120)
print("COMPARISON TABLE OVERVIEW")
print("="*120)
print(f"Total unique policies analyzed: {len(EVENTS)}")
print(f"\nTables generated for SECONDS: {sec_count}")
print(f"Tables generated for MILLISECONDS: {ms_count}")
print(f"\nAvailable comparison tables:")
for name, _, _, _ in TABLE_CONFIGS:
    print(f"  - table_{name}_sec.csv / table_{name}_ms.csv")



GENERATING TABLES FOR SECONDS

TABLE: PERFORMANCE (seconds)
                                   avg_urllc_latency  latency_violation_rate  avg_embb_throughput  avg_reward
policy                                                                                                       
Aggressive_Jam_Fixed_10                        0.147                   0.362               94.742      -2.325
Aggressive_Jam_Fixed_15                        0.148                   0.391               94.733      -2.354
Aggressive_Jam_Fixed_20                        0.148                   0.406               94.645      -2.377
Aggressive_Jam_Fixed_5                         0.147                   0.360               94.741      -2.306
Aggressive_Jam_Fixed_Baseline_0                0.138                   0.319               94.688      -2.097
Aggressive_Jam_Pattern_Decreasing              0.148                   0.381               94.715      -2.344
Aggressive_Jam_Pattern_Fixed                   0.147      